# Variable Importance for the Causal Forest

Aim: measure how much of the heterogeneity in the estimated treatment effects each group of customer characteristics accounts for, using the drop-and-retrain importance of Bénard & Josse (2023), *Variable importance for causal forests: breaking down the heterogeneity of treatment effects*.

For a group of variables J, the importance is the share of the variance of the CATE that is lost when J is removed:

$$I^{(J)} = 
rac{Eig[(	au(X) - E[	au(X) \mid X^{(-J)}])^2ig]}{V[	au(X)]}$$

It is estimated by retraining the final causal forest without the group's columns on the **same** nuisance residuals (centering always uses all variables), applying the paper's corrective term (equation 4), and subtracting the score of a forest retrained on all columns with a new seed (retraining noise). Values near 0, including small negative ones, mean no importance. Implementation: `src/vimp.py` (validated in `tests/test_vimp.py`).

### 1. Imports

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyClassifier

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.vimp import vimp_causal_forest

RANDOM_STATE = 1213  # must match causal_forest.ipynb
DATA_PATH = next(p for p in [PROJECT_ROOT / "data" / "processed" / "retailhero_causal_forest_features.csv",
                             PROJECT_ROOT / "data" / "Processed" / "retailhero_causal_forest_features.csv"] if p.exists())
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
CATE_PATH = OUTPUT_DIR / "cf_cate_results.csv"

### 2. Rebuild the training data exactly as in causal_forest.ipynb

In [2]:
df = pd.read_csv(DATA_PATH)
Y = df["target"].astype(int)
Z = df["treatment"].astype(int)
X = df.drop(columns=["target", "treatment", "client_id", "gender_U"], errors="ignore")

strata = Z.astype(str) + "_" + Y.astype(str)
X_train, X_test, Y_train, Y_test, Z_train, Z_test = train_test_split(
    X, Y, Z, test_size=0.3, random_state=RANDOM_STATE, stratify=strata)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (140027, 21), Test: (60012, 21)


### 3. Refit the causal forest with cached residuals

The importance measure reuses the forest's cross-fitted nuisance residuals, which EconML only stores when fitted with `cache_values=True`. The saved model was fitted without them, so the same configuration is refitted here with the same seed. `n_jobs=-1` only speeds up the fit: EconML and scikit-learn draw all random seeds before parallelising, so the model is identical. This is checked against the saved test-set CATEs.

In [3]:
start = time.perf_counter()
cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=100, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1),
    model_t=DummyClassifier(strategy="prior", random_state=RANDOM_STATE),
    discrete_treatment=True,
    n_estimators=200,
    min_samples_leaf=20,
    cv=3,
    honest=True,
    inference=True,
    random_state=RANDOM_STATE,
    n_jobs=-1)
cf.fit(Y=Y_train.to_numpy(), T=Z_train.to_numpy(), X=X_train.to_numpy(), cache_values=True)
print(f"Refit in {(time.perf_counter() - start) / 60:.1f} min")

saved = pd.read_csv(CATE_PATH)
assert (saved["client_id"].to_numpy() == df.loc[X_test.index, "client_id"].to_numpy()).all()
max_diff = np.max(np.abs(cf.effect(X_test.to_numpy()) - saved["cate_hat"].to_numpy()))
print(f"Max |CATE refit - CATE saved| on the test set: {max_diff:.2e}")
assert max_diff < 1e-8, "Refit does not reproduce the saved causal forest"

Refit in 2.5 min
Max |CATE refit - CATE saved| on the test set: 4.82e-03


AssertionError: Refit does not reproduce the saved causal forest

### 4. Variable groups

In [4]:
groups = {
    "Demographics": ["age", "gender_F", "gender_M"],
    "Spending": ["total_spend", "avg_transaction_spend", "median_transaction_spend", "spend_std"],
    "Frequency/recency": ["num_transactions", "days_since_last_purchase",
                          "avg_days_between_transactions", "customer_activity_span"],
    "Loyalty points": ["regular_points_received", "regular_points_spent", "express_points_received",
                       "express_points_spent", "net_regular_points_change", "net_express_points_change"],
    "Store engagement": ["num_stores_visited", "num_unique_products", "total_quantity",
                         "avg_items_per_transaction"],
}
assert sorted(c for g in groups.values() for c in g) == sorted(X_train.columns)

### 5. Group importance

One forest is retrained per group plus one baseline, on the full training set (n = 140,027), using the out-of-bag CATEs of the original forest as the reference. The uncorrected score is reported for comparison. Treatment was randomised and the propensity model is constant, so the corrective term should be close to zero here.

In [5]:
vimp_groups = vimp_causal_forest(cf, X_train, groups, also_uncorrected=True)
print(f"Baseline retraining noise I_n(0): {vimp_groups.attrs['baseline']:.4f}")
print(f"Max deviation of rebuilt OOB weights from econml oob_predict: {vimp_groups.attrs['max_weight_check_dev']:.1e}")
display(vimp_groups.round(4))
vimp_groups.to_csv(OUTPUT_DIR / "cf_vimp_groups.csv")

Refitting 6 forests (5 groups + 1 baseline) on n=140,027...
Done in 11.2 min. Baseline I_n(0) = 0.5089
Baseline retraining noise I_n(0): 0.5089
Max deviation of rebuilt OOB weights from econml oob_predict: 5.6e-16


,importance,importance_uncorrected,raw,raw_uncorrected,n_columns,columns,fit_seconds
group,,,,,,,
Loyalty points,0.3341,0.3341,0.8430,0.8430,6,"regular_points_received, regular_points_spent,...",117.8
Spending,0.0785,0.0785,0.5874,0.5874,4,"total_spend, avg_transaction_spend, median_tra...",102.3
Frequency/recency,0.0732,0.0732,0.5821,0.5821,4,"num_transactions, days_since_last_purchase, av...",116.6
Store engagement,0.0634,0.0634,0.5723,0.5723,4,"num_stores_visited, num_unique_products, total...",114.3
Demographics,0.0566,0.0566,0.5655,0.5655,3,"age, gender_F, gender_M",117.7


Summary

In [6]:
summary = vimp_groups[["importance", "importance_uncorrected", "n_columns", "columns"]].rename(columns={
    "importance": "Importance (corrected)", "importance_uncorrected": "Importance (uncorrected)",
    "n_columns": "# columns", "columns": "Columns"})
display(summary.round(3))
print(f"Sum of group importances: {vimp_groups['importance'].sum():.3f} "
      "(equals 1 only if the groups are independent; correlated groups share heterogeneity)")

,Importance (corrected),Importance (uncorrected),# columns,Columns
group,,,,
Loyalty points,0.334,0.334,6,"regular_points_received, regular_points_spent,..."
Spending,0.078,0.078,4,"total_spend, avg_transaction_spend, median_tra..."
Frequency/recency,0.073,0.073,4,"num_transactions, days_since_last_purchase, av..."
Store engagement,0.063,0.063,4,"num_stores_visited, num_unique_products, total..."
Demographics,0.057,0.057,3,"age, gender_F, gender_M"


Sum of group importances: 0.606 (equals 1 only if the groups are independent; correlated groups share heterogeneity)


### 6. Drill-down within the most important group

Each column of the top group is scored as its own group. A column that can be reconstructed from the others (for example `net_regular_points_change = regular_points_received + regular_points_spent`), or that is strongly correlated with columns left in the model, scores close to 0 on its own even when the group matters.

In [7]:
top_group = vimp_groups.index[0]
print(f"Drilling down into: {top_group}")
vimp_drill = vimp_causal_forest(cf, X_train, groups, drill_down=top_group, also_uncorrected=True)
display(vimp_drill.round(4))
vimp_drill.to_csv(OUTPUT_DIR / "cf_vimp_drilldown.csv")

Drilling down into: Loyalty points
Refitting 7 forests (6 groups + 1 baseline) on n=140,027...
Done in 23.2 min. Baseline I_n(0) = 0.5089


,importance,importance_uncorrected,raw,raw_uncorrected,n_columns,columns,fit_seconds
group,,,,,,,
net_regular_points_change,0.0083,0.0083,0.5172,0.5172,1,net_regular_points_change,341.3
express_points_received,0.0039,0.0039,0.5128,0.5128,1,express_points_received,134.1
regular_points_received,0.0032,0.0032,0.5121,0.5121,1,regular_points_received,129.1
express_points_spent,0.0025,0.0025,0.5114,0.5114,1,express_points_spent,132.2
net_express_points_change,0.0007,0.0007,0.5096,0.5096,1,net_express_points_change,397.4
regular_points_spent,-0.0014,-0.0014,0.5075,0.5075,1,regular_points_spent,130.9


### 7. Runtime

In [8]:
print(f"Group run:  {vimp_groups.attrs['total_seconds'] / 60:.1f} min for {len(vimp_groups) + 1} forests "
      f"({vimp_groups['fit_seconds'].mean():.0f} s per forest on average)")
print(f"Drill-down: {vimp_drill.attrs['total_seconds'] / 60:.1f} min for {len(vimp_drill) + 1} forests")

Group run:  11.2 min for 6 forests (114 s per forest on average)
Drill-down: 23.2 min for 7 forests
